In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ----------------------------- Config -----------------------------
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
DROP_DUPLICATES = True      # avoids train/test leakage from repeated reviews
USE_CLASS_WEIGHTS = False   # set True to boost the weak Neutral class
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_cuda = device.type == 'cuda'
print(f"Using device: {device}")

# bf16 if the GPU supports it (no scaler needed), otherwise fp16 + GradScaler
if use_cuda and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
    scaler = None
elif use_cuda:
    amp_dtype = torch.float16
    scaler = torch.amp.GradScaler('cuda')
else:
    amp_dtype = None
    scaler = None

# ------------------------- Load and preprocess -------------------------
print("\n=== Loading Data ===")
df = pd.read_csv('user_reviews.csv')
print(f"Original dataset shape: {df.shape}")

print("\n=== Preprocessing ===")
df_clean = df.dropna(subset=['Translated_Review', 'Sentiment'])
print(f"After removing NaN values: {df_clean.shape}")

df_clean = df_clean[
    (~df_clean['Translated_Review'].astype(str).str.lower().isin(['nan', 'none', ''])) &
    (~df_clean['Sentiment'].astype(str).str.lower().isin(['nan', 'none', '']))
].copy()
print(f"After removing 'nan' string values: {df_clean.shape}")

if DROP_DUPLICATES:
    df_clean = df_clean.drop_duplicates(subset=['Translated_Review']).copy()
    print(f"After removing duplicate reviews: {df_clean.shape}")

print(f"Total removed: {len(df) - len(df_clean)} rows")

print("\nSentiment Distribution:")
print(df_clean['Sentiment'].value_counts())

# Encode labels
label_encoder = LabelEncoder()
df_clean['label'] = label_encoder.fit_transform(df_clean['Sentiment'])
print(f"\nLabel Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Split data (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['Translated_Review'].values,
    df_clean['label'].values,
    test_size=0.3,
    random_state=SEED,
    stratify=df_clean['label'].values
)

print(f"\nTrain set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")


# ----------------------------- Dataset -----------------------------
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # FIX: call the tokenizer directly (encode_plus was removed in transformers v5)
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }


# ------------------------ Tokenizer and model ------------------------
# FIX: no silent random-weights fallback; if loading fails, the error surfaces.
print("\n=== Initializing BERT Model ===")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
num_labels = len(label_encoder.classes_)
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)
model = model.to(device)
print("Model loaded successfully!")

# Datasets and dataloaders
train_dataset = SentimentDataset(X_train, y_train, tokenizer, MAX_LENGTH)
test_dataset = SentimentDataset(X_test, y_test, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False)

# Optimizer + scheduler (linear warmup/decay)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Loss (optionally class-weighted)
if USE_CLASS_WEIGHTS:
    weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    print(f"Class weights: {dict(zip(label_encoder.classes_, weights.round(3)))}")
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))
else:
    loss_fn = nn.CrossEntropyLoss()


# ----------------------------- Training -----------------------------
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(data_loader, desc="Training")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        if amp_dtype is not None:
            with torch.autocast(device_type='cuda', dtype=amp_dtype):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = loss_fn(outputs.logits.float(), labels)
        else:
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)

        logits = outputs.logits

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()

        total_loss += loss.item()
        predicted = torch.argmax(logits, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix({'loss': loss.item(), 'acc': correct / total})

    return total_loss / len(data_loader), correct / total


def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            if amp_dtype is not None:
                with torch.autocast(device_type='cuda', dtype=amp_dtype):
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    return predictions, actual_labels


print("\n=== Training BERT Model ===")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")

# ---------------------------- Evaluation ----------------------------
print("\n=== Evaluating on Test Set ===")
predictions, actual_labels = evaluate(model, test_loader, device)

accuracy = accuracy_score(actual_labels, predictions)
print(f"\nTest Accuracy: {accuracy:.4f}")

print("\n=== Classification Report ===")
print(classification_report(
    actual_labels,
    predictions,
    target_names=label_encoder.classes_,
    digits=4
))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(actual_labels, predictions))

# ------------------------------ Save ------------------------------
print("\n=== Saving Model ===")
model.save_pretrained('./app_review_sentiment_bert')
tokenizer.save_pretrained('./app_review_sentiment_bert')
print("Model saved to './app_review_sentiment_bert'")

# ----------------------- Example predictions -----------------------
print("\n=== Example Predictions ===")
model.eval()
for i, text in enumerate(X_test[:5]):
    # FIX: call the tokenizer directly here too
    encoding = tokenizer(
        str(text),
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1)

    predicted_sentiment = label_encoder.inverse_transform([pred.cpu().item()])[0]
    actual_sentiment = label_encoder.inverse_transform([y_test[i]])[0]

    print(f"\nText: {str(text)[:100]}...")
    print(f"Predicted: {predicted_sentiment} | Actual: {actual_sentiment}")

print("\n=== Training Complete ===")

Using device: cuda

=== Loading Data ===
Original dataset shape: (64295, 5)

=== Preprocessing ===
After removing NaN values: (37427, 5)
After removing 'nan' string values: (37427, 5)
After removing duplicate reviews: (27994, 5)
Total removed: 36301 rows

Sentiment Distribution:
Sentiment
Positive    17593
Negative     6240
Neutral      4161
Name: count, dtype: int64

Label Mapping: {'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}

Train set size: 19595
Test set size: 8399

=== Initializing BERT Model ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!

=== Training BERT Model ===

Epoch 1/3


Training: 100%|██████████| 1225/1225 [10:17<00:00,  1.98it/s, loss=0.0297, acc=0.813]


...

Train Loss: 0.1214, Train Accuracy: 0.9691

=== Evaluating on Test Set ===


Evaluating: 100%|██████████| 263/263 [01:26<00:00,  3.04it/s]



Test Accuracy: 0.9440

=== Classification Report ===
              precision    recall  f1-score   support

    Negative     0.8936    0.9103    0.9018      1872
     Neutral     0.9508    0.8975    0.9234      1249
    Positive     0.9607    0.9670    0.9638      5278

    accuracy                         0.9440      8399
   macro avg     0.9350    0.9249    0.9297      8399
weighted avg     0.9442    0.9440    0.9440      8399


=== Confusion Matrix ===
[[1704   25  143]
 [  62 1121   66]
 [ 141   33 5104]]

=== Saving Model ===


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to './app_review_sentiment_bert'

=== Example Predictions ===

Text: This nice I like interesting...
Predicted: Positive | Actual: Positive

Text: Made rich...
Predicted: Positive | Actual: Positive

Text: I'm can't go site...
Predicted: Neutral | Actual: Neutral

Text: I really like since got phone tons pop ads day...
Predicted: Positive | Actual: Positive

Text: It fun game gg/ good game. Has fun things get. You buy places I already Bakery first I level 20 fun ...
Predicted: Positive | Actual: Positive

=== Training Complete ===


In [6]:
model.save_pretrained("./bert_sentiment_model")
tokenizer.save_pretrained("./bert_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_sentiment_model/tokenizer_config.json',
 './bert_sentiment_model/tokenizer.json')

In [7]:
!ls -lh ./bert_sentiment_model

total 419M
-rw-r--r-- 1 root root  920 Oct  6 15:04 config.json
-rw------- 1 root root 418M Oct  6 15:04 model.safetensors
-rw-r--r-- 1 root root  351 Oct  6 15:04 tokenizer_config.json
-rw-r--r-- 1 root root 695K Oct  6 15:04 tokenizer.json


In [8]:
!zip -r bert_sentiment_model.zip bert_sentiment_model

  adding: bert_sentiment_model/ (stored 0%)
  adding: bert_sentiment_model/tokenizer.json (deflated 71%)
  adding: bert_sentiment_model/config.json (deflated 54%)
  adding: bert_sentiment_model/model.safetensors (deflated 7%)
  adding: bert_sentiment_model/tokenizer_config.json (deflated 43%)


In [10]:
from google.colab import files

files.download("bert_sentiment_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>